# Lineup Construction

## tl;dr
16,752 five-player lineups expand to 83,760 core/fifth representations.

## Context & Methods
IDs are numerically sorted, with team and season added to contextual keys. The same outcome is deliberately represented five ways.

### Key Assumptions
Regular-season PBP Stats aggregates; lagged profiles; observational comparisons. See [methodology](../docs/methodology_notes.md).

## Data
Run the cached pipeline and offline analysis before this notebook. No network requests are made here.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'data_pipeline.py').exists())
DATA = ROOT / 'data' / 'processed'
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
pd.set_option('display.max_columns', 12)


## Results

### Validate original and expanded keys

In [2]:
lineups=pd.read_csv(DATA/'lineups.csv')
cores=pd.read_csv(DATA/'core_observations_unfiltered.csv')
assert len(cores)==5*len(lineups)
assert not lineups.lineup_key.duplicated().any()
assert not cores.duplicated(['core_key','fifth_id']).any()
display(cores[['season','team','core_id','fifth_id','lineup_id','possessions']].head())

,season,team,core_id,fifth_id,lineup_id,possessions
0,2022-23,BOS,203935-1627759-1628369-1628401,201143,201143-203935-1627759-1628369-1628401,898.5
1,2022-23,BOS,201143-1627759-1628369-1628401,203935,201143-203935-1627759-1628369-1628401,898.5
2,2022-23,BOS,201143-203935-1628369-1628401,1627759,201143-203935-1627759-1628369-1628401,898.5
3,2022-23,BOS,201143-203935-1627759-1628401,1628369,201143-203935-1627759-1628369-1628401,898.5
4,2022-23,BOS,201143-203935-1627759-1628369,1628401,201143-203935-1627759-1628369-1628401,898.5


### Trace one real lineup

In [3]:
key=lineups.sort_values('possessions',ascending=False).iloc[0].lineup_key
display(cores[cores.lineup_key==key][['core_id','fifth_id','lineup_name','possessions','net_rating']])

,core_id,fifth_id,lineup_name,possessions,net_rating
16985,1627734-1628368-1628989-1631099,203084,"Domantas Sabonis, De'Aaron Fox, Kevin Huerter,...",1876.5,0.868224
16986,203084-1628368-1628989-1631099,1627734,"Domantas Sabonis, De'Aaron Fox, Kevin Huerter,...",1876.5,0.868224
16987,203084-1627734-1628989-1631099,1628368,"Domantas Sabonis, De'Aaron Fox, Kevin Huerter,...",1876.5,0.868224
16988,203084-1627734-1628368-1631099,1628989,"Domantas Sabonis, De'Aaron Fox, Kevin Huerter,...",1876.5,0.868224
16989,203084-1627734-1628368-1628989,1631099,"Domantas Sabonis, De'Aaron Fox, Kevin Huerter,...",1876.5,0.868224


### Reconcile counts without duplication

In [4]:
display(pd.DataFrame({'lineup_rows':[len(lineups)],'expanded_rows':[len(cores)],'lineup_exposure':[lineups.possessions.sum()],'expanded_exposure_divided_by_five':[cores.possessions.sum()/5]}))

,lineup_rows,expanded_rows,lineup_exposure,expanded_exposure_divided_by_five
0,16752,83760,244628.0,244628.0


## Takeaways
Expansion enables same-core comparisons; it does not create new minutes or independent evidence. Regression weights divide exposure among retained representations.